# garage door simulator

python plays the garage door for the OpenPLC garage-door program.

it reads the motor outputs from the PLC over modbus, moves a pretend door, and writes the limit switches back. same job as the animated door in LogixPro.

```
OpenPLC (your ladder)  --motor_up / motor_down-->  this notebook
this notebook          --top_limit / bottom_limit-->  OpenPLC
```

before you start:
- the OpenPLC runtime is running (`docker start openplc-runtime`, or `docker compose up -d`)
- the garage-door program is uploaded and the PLC is started in the Editor
- kernel is the `.venv` for this folder (or you're in the docker notebooks container)

## 1. settings

**connection**: on your mac the PLC is at `127.0.0.1`. inside docker, compose sets `PLC_HOST=openplc`. you don't need to change anything.

**addresses**: these match the variables in your ladder. coils for `%QX` start at 0. coils for `%MX` start right after the 8192 `%QX` coils, so `%MX0.0` is 8192.

| python name | modbus coil | ladder variable | who writes it |
|---|---|---|---|
| MOTOR_UP | 0 | motor_up `%QX0.0` | PLC |
| MOTOR_DOWN | 1 | motor_down `%QX0.1` | PLC |
| TOP_LIMIT | 8192 | top_limit `%MX0.0` | python |
| BOTTOM_LIMIT | 8193 | bottom_limit `%MX0.1` | python |

**door behavior**: how long the door takes to travel, and how often python checks the PLC.

In [ ]:
import os
import time
from pymodbus.client import ModbusTcpClient

# connection
HOST = os.environ.get("PLC_HOST", "127.0.0.1")
PORT = int(os.environ.get("PLC_PORT", "1502"))

# modbus coil addresses
MOTOR_UP = 0          # %QX0.0  (plc writes, python reads)
MOTOR_DOWN = 1        # %QX0.1  (plc writes, python reads)
TOP_LIMIT = 8192      # %MX0.0  (python writes, plc reads)
BOTTOM_LIMIT = 8193   # %MX0.1  (python writes, plc reads)

# door behavior
TRAVEL_SECONDS = 5.0  # time for the door to go fully up or down
STEP = 0.1            # how often python checks the plc, in seconds

print(f"plc at {HOST}:{PORT}")

## 2. helper functions

- `read_bit()` reads one coil from the PLC
- `write_bit()` writes one coil to the PLC
- `door_bar()` draws the door as a bar, 0% is shut and 100% is open

In [ ]:
def read_bit(client, address):
    result = client.read_coils(address, count=1)
    if result.isError():
        raise RuntimeError(f"could not read coil {address}: {result}")
    return bool(result.bits[0])


def write_bit(client, address, value):
    result = client.write_coil(address, value)
    if result.isError():
        raise RuntimeError(f"could not write coil {address}: {result}")


def door_bar(position):
    filled = int(position / 5)
    return "[" + "#" * filled + "." * (20 - filled) + f"] {position:5.1f}% open"


print(door_bar(0))
print(door_bar(50))
print(door_bar(100))

## 3. check the connection

reads the motor outputs once and stops. if this fails, fix it before step 4:
- `could not connect`: is the runtime running? (`docker ps` should list it)
- `could not read coil`: is the program uploaded and the PLC started in the Editor?

In [ ]:
client = ModbusTcpClient(HOST, port=PORT)
if client.connect():
    print("connected")
    print("motor_up:  ", read_bit(client, MOTOR_UP))
    print("motor_down:", read_bit(client, MOTOR_DOWN))
else:
    print(f"could not connect to {HOST}:{PORT}. is the plc running with the modbus server on?")
client.close()

## 4. run the door

this cell keeps running until you stop it with **Interrupt** (the ■ in the notebook toolbar).

every 0.1 seconds it:
1. reads `motor_up` and `motor_down`
2. moves the door up or down a little
3. writes `top_limit` (door at 100%) and `bottom_limit` (door at 0%)

the door starts shut, so `bottom_limit` goes TRUE right away and `shut_lamp` lights.
press open in the OpenPLC debugger, or run step 10 in dock_shipments.ipynb, and watch the bar fill.

In [ ]:
def run_door():
    client = ModbusTcpClient(HOST, port=PORT)
    if not client.connect():
        print(f"could not connect to {HOST}:{PORT}. is the plc running with the modbus server on?")
        return

    position = 0.0  # door starts shut
    speed = 100.0 / TRAVEL_SECONDS * STEP
    print("connected. door starts shut. press open in the openplc debugger. interrupt to stop.")

    try:
        while True:
            up = read_bit(client, MOTOR_UP)
            down = read_bit(client, MOTOR_DOWN)

            if up and not down:
                position = min(100.0, position + speed)
            elif down and not up:
                position = max(0.0, position - speed)

            write_bit(client, TOP_LIMIT, position >= 100.0)
            write_bit(client, BOTTOM_LIMIT, position <= 0.0)

            motor = "UP  " if up else "DOWN" if down else "stop"
            print(f"\rmotor {motor}  {door_bar(position)}", end="", flush=True)
            time.sleep(STEP)
    except KeyboardInterrupt:
        print("\nstopped.")
    except Exception as err:
        print(f"\nlost the plc connection: {err}")
    finally:
        client.close()


run_door()